### Mount drive and load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/drive/MyDrive/AML-Project/market.zip -d /content/Market-Pytorch

### Load libraries

In [4]:
import torch
import torch.nn as nn
from torch.nn import init
import torch.optim as optim
from torchvision import models
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

import os
import shutil
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import timm
import json
import numpy as np
import faiss
from PIL import Image
import copy
import random
import itertools

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(12)
np.random.seed(12)
random.seed(12)

# LA Transformers (Pretrained: ImageNet | Fine tuning: Market-1501)

**PERSON RE-IDENTIFICATION 101**

The concept is rather simple. We have N unique persons in our dataset (N classes). Each of those has M (where M can change among the different classes, but let's simplify) images of the same person from different angles.

The main idea is to have a network that can extract very similar features for the same person images taken from different angles. In this way, one could just do a dot product over the extracted features of two images (or equal similarity measures for tensors) to check if they correspond to same person (high measurement, same person, lower, different).

**How do we achieve this?**

How can we train a network that can extract equal features for same person from different angles?

Let's delve into one of the most basic approaches (this is the approach of LA Transformers, but our idea was also to test contrastive methods after this).

We train a classifier composed of a backbone to extract features (e.g. CNN, ViT, ...) + final linear layer (the final linear layer is often referred to as the *head*). We pass to this network the images of the persons and we train to classify them into their class (there is a unique number associated with each person image, remember that, as said above, there are N*M images and N classes). In this way, the network, for the same class will see the images of the same person but taken from all different angles.
The idea is that in this way the backbone will be forced to output almost exact same features for different camera angle images containing the same person (that is, different images belonging to same class).

Finally, at test time we will detach the head, and just extract features. So for a given image (a query) we will extract features of it. Then we will do the same of a bunch of K images (gallery) and measure the similarity between query features and each of the gallery images features.

We will end up with a tensor of K values, where the highest one represent the best match between the query and that particular gallery image.

****

The idea behind Locally Aware Transformers in simple but effective. Basically, since BERT, to do classification over sequences a special token is appended at the start of the sequence. This token, called CLS, contains global informations about the data we are processing (in our case the image).

The underlying concept here is used is the following:
- append CLS token at start of sequence like one would normally do in classifying sequences using transformers
- ViT output a sequence of patches computed from the image (of shape BATCH, SEQ_LEN, PATCH_DIM)
- Average over the SEQ_LEN dimension, so that, no matter how big the image is (and so no matter how big is SEQ_LEN), we will end up always with a fixed size value for that dimension
- For each patch, that represent local informations of the image with respect to that patch, embedd also the CLS token, that is, the global information. In this way we are enriching local informations with global ones (this is done with a weighted average over the extracted patches and the CLS token. The weight of the patches is specified by the value lmbd)
- For each enriched patch, pass it through a simple classifier that outputs the probability distribution over the N classes (that is, the N different persons in the dataset)
- In the training loop the probability distributions obtained from the different patches are summed together. Then, the maximum entry is taken as prediction

In [13]:
def weights_init_kaiming(m):
    classname = m.__class__.__name__
    # print(classname)
    if classname.find('Conv') != -1:
        init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
    elif classname.find('Linear') != -1:
        init.kaiming_normal_(m.weight.data, a=0, mode='fan_out')
        init.constant_(m.bias.data, 0.0)
    elif classname.find('BatchNorm1d') != -1:
        init.normal_(m.weight.data, 1.0, 0.02)
        init.constant_(m.bias.data, 0.0)

def weights_init_classifier(m):
    classname = m.__class__.__name__
    if classname.find('Linear') != -1:
        init.normal_(m.weight.data, std=0.001)
        init.constant_(m.bias.data, 0.0)

class ClassBlock(nn.Module):
    def __init__(self, input_dim, class_num, droprate, relu=False, bnorm=True, num_bottleneck=512, linear=True, return_f = False):
        super(ClassBlock, self).__init__()
        self.return_f = return_f
        add_block = []
        if linear:
            add_block += [nn.Linear(input_dim, num_bottleneck)]
        else:
            num_bottleneck = input_dim
        if bnorm:
            add_block += [nn.BatchNorm1d(num_bottleneck)]
        if relu:
            add_block += [nn.LeakyReLU(0.1)]
        if droprate>0:
            add_block += [nn.Dropout(p=droprate)]
        add_block = nn.Sequential(*add_block)
        add_block.apply(weights_init_kaiming)

        classifier = []
        classifier += [nn.Linear(num_bottleneck, class_num)]
        classifier = nn.Sequential(*classifier)
        classifier.apply(weights_init_classifier)

        self.add_block = add_block
        self.classifier = classifier
    def forward(self, x):
        x = self.add_block(x)
        if self.return_f:
            f = x
            x = self.classifier(x)
            return [x,f]
        else:
            x = self.classifier(x)
            return x

class LATransformer(nn.Module):
    def __init__(self, model, lmbd, print_verbose = False, test=False, pretraining=False):
        super(LATransformer, self).__init__()

        if print_verbose:
            self._print = print
        else:
            self._print = lambda *args, **kwargs: None
        self.class_num = 751
        self.part = 14 # We cut the pool5 to sqrt(N) parts
        self.num_blocks = 12
        self.model = model
        self.model.head.requires_grad_ = False
        self.cls_token = self.model.cls_token
        self.pos_embed = self.model.pos_embed
        self.avgpool = nn.AdaptiveAvgPool2d((self.part,768))
        self.dropout = nn.Dropout(p=0.5)
        self.lmbd = lmbd
        self.test = test
        self.pretraining = pretraining
        if not (self.test or self.pretraining):
          for i in range(self.part):
              name = 'classifier'+str(i)
              setattr(self, name, ClassBlock(768, self.class_num, droprate=0.5, relu=False, bnorm=True, num_bottleneck=256))

        if self.pretraining:
          self.fc = nn.Sequential(nn.Conv1d(14, 32, 3),
                                  nn.BatchNorm1d(32),
                                  nn.LeakyReLU(0.1),
                                  nn.Conv1d(32, 3, 3),
                                  nn.BatchNorm1d(3),
                                  nn.LeakyReLU(0.1),
                                  nn.Flatten(),
                                  nn.Linear(2292, 1024),
                                  nn.LeakyReLU(0.1),
                                  nn.Linear(1024,128))

          self.fc.apply(weights_init_kaiming)



    def forward(self,x):

        # Divide input image into patch embeddings and add position embeddings
        # cls token is a learnable parameter added to the start of sequence
        # It contains global info about the whole image
        # Used in classical Transformers like BERT and ViT to do classification with just itself
        # Here it is later combined to enrich local features with global features of the image
        self._print(f"x before pos embedding: {x.shape}")
        x = self.model.patch_embed(x)
        self._print(f"x after pos embedding: {x.shape}")
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        self._print(f"cls token: {cls_token.shape}")
        x = torch.cat((cls_token, x), dim=1)
        self._print(f"x with concatenation with cls token: {x.shape}")
        x = self.model.pos_drop(x + self.pos_embed)
        self._print(f"x after pos drop: {x.shape}")

        # Feed forward through transformer blocks
        for i in range(self.num_blocks):
            self._print(f"x before block {i}: {x.shape}")
            x = self.model.blocks[i](x)
        x = self.model.norm(x)
        self._print(f"x after blocks: {x.shape}")

        # extract the cls token
        cls_token_out = x[:, 0].unsqueeze(1)
        self._print(f"cls token out: {cls_token_out.shape}")

        # Average pool
        x = self.avgpool(x[:, 1:])
        self._print(f"x after avgpool: {x.shape}")

        if self.test:
          return x

        # Add global cls token to each local token
        for i in range(self.part):
            self._print(f"x before mul: {x.shape}")
            out = torch.mul(x[:, i, :], self.lmbd)
            x[:,i,:] = torch.div(torch.add(cls_token_out.squeeze(),out), 1+self.lmbd)

        if self.pretraining:
          x = x.reshape(x.size(0), 14, -1)
          x = self.fc(x)
          return x

        # Locally aware network
        part = {}
        predict = {}
        for i in range(self.part):
            part[i] = x[:,i,:]
            name = 'classifier'+str(i)
            c = getattr(self,name)
            predict[i] = c(part[i])
        return predict

**How the dataset is composed?**
The dataset (just the things we are using here) has the following structure:
- "train" is a folder with N subfolders. Each subfolder has a number, identifying the class to which the images inside the folder belong. A class is just a number identifying a person inside the images taken from the different cameras perspectives
- "val" has the same structure as train, with same classes (that is subfolders names) but different images for each subclass with respect to the same class in the "train" folder
- "gallery" again has same structure but completely different classes (it makes sense, we now don't want to see the same images seen at training time)
- "query" has same structure and classes of gallery, with just different images.

While it make sense to have same classes for query and gallery, because given a query image (an image taken from a particular camera) we want to check among the gallery ones (images of persons included query from various camera angles) for the best match, it could be dubious to have same classes in train and val.

To clear up any doubts, remember that, as explained above, there is a fundamental difference between training and testing. At training time we are doing classification, while at test time we are doing feature matching. Because of this, during training, to have a meaningful validation set, we need same classes just different images (like one would normally do in image classification).

In [ ]:
def get_data(data_dir="/content/Market-Pytorch/Market"):
  batch_size = 32

  transform_train_list = transforms.Compose([
      transforms.Resize((224,224), interpolation=3),
      transforms.RandomHorizontalFlip(),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
      ])
  transform_val_list = transforms.Compose([
      transforms.Resize(size=(224,224),interpolation=3),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
      ])

  dataset_train = datasets.ImageFolder(os.path.join(data_dir, 'train'),transform=transform_train_list)
  dataset_val = datasets.ImageFolder(os.path.join(data_dir, 'val'),transform=transform_val_list)
  train_loader = DataLoader(dataset = dataset_train, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(dataset = dataset_val, batch_size=batch_size, shuffle=True)

  return train_loader, val_loader

In [ ]:
def freeze_all_blocks(model):
    frozen_blocks = 12
    for block in model.model.blocks[:frozen_blocks]:
        for param in block.parameters():
            param.requires_grad=False

def unfreeze_blocks(model, amount= 1):
    for block in model.model.blocks[11-amount:]:
        for param in block.parameters():
            param.requires_grad=True
    return model

def get_training_objects(pretraining_path = None):
  backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=751).to(device)
  lmbd = 8 #Weight in averaging with CLS token
  model = LATransformer(backbone, lmbd).to(device)

  if pretraining_path:
    model.load_state_dict(torch.load(pretraining_path), strict=False)

  freeze_all_blocks(model)

  criterion = nn.CrossEntropyLoss()

  optimizer = optim.AdamW(model.parameters(),weight_decay=5e-4, lr=3e-4)

  return model, criterion, optimizer

Check above markdown cells to understand what's going on in the training loop. Overall is a classic training loop, nothing fancy, but score computation changes with respect to a classical image classification task.

In [ ]:
def train(num_epochs, model, train_loader, val_loader, optimizer, loss_fn, save_model_path, log_file):

    os.makedirs(save_model_path, exist_ok=True)
    log_data = []
    unfrozen_blocks = 0
    unfreeze_after = 2
    lr_decay = .8

    with tqdm(total=num_epochs, desc='Total Progress', unit='epoch') as epoch_pbar:
      for epoch in range(num_epochs):

          if epoch%unfreeze_after==0:
            unfrozen_blocks += 1
            model = unfreeze_blocks(model, unfrozen_blocks)
            optimizer.param_groups[0]['lr'] *= lr_decay
            trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print("Unfrozen Blocks: {}, Current lr: {}, Trainable Params: {}".format(unfrozen_blocks,
                                                                                optimizer.param_groups[0]['lr'],
                                                                                trainable_params))

          model.train()
          train_loss = 0.0
          train_accuracy = 0.0

          with tqdm(train_loader, unit="batch") as pbar:
              for data, target in pbar:
                  data, target = data.to(device), target.to(device)

                  optimizer.zero_grad()
                  output = model(data)

                  score = sum(nn.Softmax(dim=1)(v) for v in output.values())
                  _, preds = torch.max(score, 1)

                  loss = sum(loss_fn(v, target) for v in output.values())
                  loss.backward()
                  optimizer.step()

                  train_loss += loss.item()
                  train_accuracy += (preds == target).float().mean().item()

                  current_loss = train_loss / (pbar.n + 1)
                  current_accuracy = train_accuracy / (pbar.n + 1)

                  pbar.set_description(f"Epoch {epoch+1}/{num_epochs}")
                  pbar.set_postfix(loss=current_loss, accuracy=current_accuracy)

          avg_train_loss = train_loss / len(train_loader)
          avg_train_accuracy = train_accuracy / len(train_loader)

          model.eval()
          val_loss = 0.0
          val_accuracy = 0.0

          with torch.no_grad():
              with tqdm(val_loader, unit="batch") as val_pbar:
                  for data, target in val_pbar:
                      data, target = data.to(device), target.to(device)

                      output = model(data)

                      score = sum(nn.Softmax(dim=1)(v) for v in output.values())
                      _, preds = torch.max(score, 1)

                      loss = sum(loss_fn(v, target) for v in output.values())

                      val_loss += loss.item()
                      val_accuracy += (preds == target).float().mean().item()

                      current_loss = val_loss / (val_pbar.n + 1)
                      current_accuracy = val_accuracy / (val_pbar.n + 1)

                      val_pbar.set_description(f"Epoch {epoch+1}/{num_epochs} [Val]")
                      val_pbar.set_postfix(loss=current_loss, accuracy=current_accuracy)

          avg_val_loss = val_loss / len(val_loader)
          avg_val_accuracy = val_accuracy / len(val_loader)

          model_save_path = os.path.join(save_model_path, f'model_epoch_{epoch+1}.pth')
          torch.save(model.state_dict(), model_save_path)

          epoch_log = {
              'epoch': epoch + 1,
              'training_loss': avg_train_loss,
              'training_accuracy': avg_train_accuracy,
              'val_loss': avg_val_loss,
              'val_accuracy': avg_val_accuracy
          }
          log_data.append(epoch_log)

          with open(log_file, 'w') as f:
              json.dump(log_data, f, indent=4)

          print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {avg_train_loss:.4f} - Validation Loss: {avg_val_loss:.4f} - Training Accuracy: {avg_train_accuracy:.4f} - Validation Accuracy: {avg_val_accuracy:.4f}")

          epoch_pbar.update(1)
          epoch_pbar.set_postfix(
                train_loss=avg_train_loss,
                train_accuracy=avg_train_accuracy,
                val_loss=avg_val_loss,
                val_accuracy=avg_val_accuracy
          )
          print("==================================================================================")

In [ ]:
def train_model(save_path = "/content/drive/MyDrive/AML-Project/LA-Transformers(Vanilla)/", pretraining_path = None):
  train_loader, val_loader = get_data()
  model, criterion, optimizer = get_training_objects(pretraining_path)
  train(30, model, train_loader, val_loader, optimizer, criterion, save_path, os.path.join(save_path, "training_results.json"))

In [ ]:
train_model()